In [1]:
!pip install pennylane scikit-learn matplotlib -q

import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 112.6 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
# 2. تحميل وتجهيز بيانات Iris
iris = load_iris()
X = iris.data
y = iris.target

In [3]:
# تصفية البيانات لتشمل كلاسين فقط (Binary Classification) لتسهيل التدريب الكوانتي
X = X[y != 2]
y = y[y != 2]
y = np.where(y == 0, -1, 1)  # تحويل الفئات إلى -1 و 1 لتتناسب مع قياسات PauliZ

In [4]:
# تقسيم البيانات وتطبيعها
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
# 3. بناء الدائرة الكوانتية (Quantum Circuit Setup)
num_qubits = 4
dev = qml.device("default.qubit", wires=num_qubits)

@qml.qnode(dev)
def quantum_circuit(weights, features):
    # إدخال الميزات في الدائرة الكوانتية (Feature Map)
    qml.AngleEmbedding(features, wires=range(num_qubits))

    # الطبقات القابلة للتدريب (Variational Circuit)
    qml.StronglyEntanglingLayers(weights, wires=range(num_qubits))

    # إرجاع قيمة القياس على الـ Qubit الأول
    return qml.expval(qml.PauliZ(0))

# 4. تعريف دالة الخسارة والدقة (Loss & Accuracy)
def Variational_Classifier(weights, bias, features):
    return quantum_circuit(weights, features) + bias

def cost(weights, bias, X, y):
    predictions = [Variational_Classifier(weights, bias, x) for x in X]
    return np.mean((y - predictions) ** 2)

def accuracy(weights, bias, X, y):
    predictions = [np.sign(Variational_Classifier(weights, bias, x)) for x in X]
    return np.mean(predictions == y)

In [11]:
# 5. تهيئة الأوزان والتحسين (Optimization)
num_layers = 2
np.random.seed(42)
weights = 0.01 * np.random.randn(num_layers, num_qubits, 3, requires_grad=True)
bias = np.array(0.0, requires_grad=True)

opt = qml.AdamOptimizer(stepsize=0.1)
epochs = 25

print("--- بدء تدريب النموذج الكوانتي ---")
for epoch in range(epochs):
    # Corrected opt.step call: X_train and y_train are captured by the lambda closure
    # and passed to cost, not directly as optimizable arguments to opt.step.
    # Correctly unpack only the two returned parameters (weights, bias).
    weights, bias = opt.step(lambda w, b: cost(w, b, X_train, y_train), weights, bias)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        current_cost = cost(weights, bias, X_train, y_train)
        acc_train = accuracy(weights, bias, X_train, y_train)
        print(f"Epoch {epoch + 1:2d} | Cost: {current_cost:.4f} | Train Accuracy: {acc_train * 100:.2f}%")

--- بدء تدريب النموذج الكوانتي ---


/usr/local/lib/python3.13/dist-packages/autograd/tracer.py:16: UserWarning: Output seems independent of input.
  warnings.warn("Output seems independent of input.")


Epoch  1 | Cost: 1.0205 | Train Accuracy: 50.00%
Epoch  5 | Cost: 1.0205 | Train Accuracy: 50.00%
Epoch 10 | Cost: 1.0205 | Train Accuracy: 50.00%
Epoch 15 | Cost: 1.0205 | Train Accuracy: 50.00%
Epoch 20 | Cost: 1.0205 | Train Accuracy: 50.00%
Epoch 25 | Cost: 1.0205 | Train Accuracy: 50.00%


In [14]:
# 6. تقييم النموذج على بيانات الاختبار
acc_test = accuracy(weights, bias, X_test, y_test)
print(f"\nTest Accuracy: {acc_test * 100:.2f}%")


Test Accuracy: 30.00%


الان سوف درب النموذج باتستعمل خوارزميه ال svc
لنقوم بعدها بالمقارنه بين التدريبين

In [15]:
# 1. تثبيت المكتبات
!pip install pennylane scikit-learn matplotlib -q

import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [16]:
# 2. تحميل وتجهيز البيانات
iris = load_iris()
X = iris.data
y = iris.target

# تصنيف ثنائي (Binary Classification)
X = X[y != 2]
y = y[y != 2]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [17]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [19]:
#3. بناء الدائرة الكوانتية لحساب مصفوفة النواة (Quantum Kernel Computation)
num_qubits = 4
dev = qml.device("default.qubit", wires=num_qubits)

@qml.qnode(dev)
def kernel_circuit(x1, x2):
    # ترميز العينة الأولى (Encoding x1)
    qml.AngleEmbedding(x1, wires=range(num_qubits))
    # العكس لترميز العينة الثانية (Adjoint Encoding x2)
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(num_qubits))
    # قياس الاحتمالية للوصول للحالة |0..0>
    return qml.probs(wires=range(num_qubits))

def quantum_kernel_matrix(A, B):
    """حساب مصفوفة النواة الكوانتية بين مجموعتين من البيانات"""
    matrix = np.zeros((len(A), len(B)))
    for i, x1 in enumerate(A):
        for j, x2 in enumerate(B):
            # القيمة في أعلى الاحتمالات تعبر عن الفروقات والتداخل (Overlap)
            matrix[i, j] = kernel_circuit(x1, x2)[0]
    return matrix

In [20]:
print("--- جاري حساب مصفوفة النواة الكوانتية (Quantum Kernel) ---")
K_train = quantum_kernel_matrix(X_train, X_train)
K_test = quantum_kernel_matrix(X_test, X_train)

# 4. تدريب خوارزمية SVC بالاعتماد على النواة الكوانتية (QSVC)
qsvc = SVC(kernel="precomputed")
qsvc.fit(K_train, y_train)

--- جاري حساب مصفوفة النواة الكوانتية (Quantum Kernel) ---


SVC(kernel='precomputed')

In [21]:
# 5. التنبؤ وتقييم الأداء
y_pred_train = qsvc.predict(K_train)
y_pred_test = qsvc.predict(K_test)

train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)

print(f"\nدقة التدريب (Train Accuracy): {train_acc * 100:.2f}%")
print(f"دقة الاختبار (Test Accuracy): {test_acc * 100:.2f}%")


دقة التدريب (Train Accuracy): 100.00%
دقة الاختبار (Test Accuracy): 100.00%
